DATASET: [DBpedia Ontology](https://www.kaggle.com/datasets/thedevastator/dbpedia-ontology-dataset)

# Iteradores y Generadores

In [ ]:
lista = [1,2,3]
tupla = (1,2,3)
cadena = "Python"

for elemento in cadena:
  print(elemento)

P
y
t
h
o
n


In [ ]:
lista = [1,2,3]
iterador = iter(lista)
iterador

In [ ]:
print(next(iterador))

1


In [ ]:
def contador_generador(max_numero):
  numero = 0
  while numero < max_numero:
    numero += 1
    yield numero

In [ ]:
for num in contador_generador(3):
  print(num)

In [ ]:
def contador():
  yield 1
  yield 2
  yield 3

In [ ]:
gen = contador()
print(next(gen))

# Preprocesamiento usando TorchText

In [ ]:
!pip install portalocker==2.7.0
!pip install torchtext==0.15.1

In [ ]:
import torch
import torchtext

In [ ]:
import pandas as pd
df_train = pd.read_csv("/content/train.csv")
df_test = pd.read_csv("/content/test.csv")

In [ ]:
df_train.head(1)

,label,title,content
0,0,E. D. Abbott Ltd,Abbott of Farnham E D Abbott Limited was a Br...


# Convertir DataFrames en Iterables

In [ ]:
train_iter = iter(df_train.itertuples(index=False, name=None))
next(train_iter)

(0,
 'E. D. Abbott Ltd',
 ' Abbott of Farnham E D Abbott Limited was a British coachbuilding business based in Farnham Surrey trading under that name from 1929. A major part of their output was under sub-contract to motor vehicle manufacturers. Their business closed in 1972.')

In [ ]:
test_iter = iter(df_test.itertuples(index=False, name=None))
next(test_iter)

(0,
 'TY KU',
 " TY KU /taɪkuː/ is an American alcoholic beverage company that specializes in sake and other spirits. The privately-held company was founded in 2004 and is headquartered in New York City New York. While based in New York TY KU's beverages are made in Japan through a joint venture with two sake breweries. Since 2011 TY KU's growth has extended its products into all 50 states.")

# Tokenizador

In [ ]:
from torchtext.data.utils import get_tokenizer
tokenizador = get_tokenizer("basic_english")

In [ ]:
token_list = tokenizador("E. D. Abbott i")
token_list

['e', '.', 'd', '.', 'abbott', 'i']

# Crear Vocabulario

In [ ]:
from torchtext.vocab import build_vocab_from_iterator

def yield_tokens(data_iter):
  for row in data_iter:
    yield tokenizador(row[1])

vocab = build_vocab_from_iterator(yield_tokens(train_iter), specials=["<unk>"])
vocab.set_default_index(vocab["<unk>"])

In [ ]:
vocab(tokenizador("Hello World!"))

[2673, 88, 41]

# Dataloader

In [ ]:
texto_pipeline = lambda x: vocab(tokenizador(x))
label_pipeline = lambda x: int(x) - 1

print(texto_pipeline("Hello World!"))
print(label_pipeline("1"))

[2673, 88, 41]
0


In [ ]:
device = torch.device("cuda" if  torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
def collate_batch(batch):
  label_list = []
  text_list = []
  offset = [0]

  for (_label, _text) in batch:
    label_list.append(label_pipeline(_label))
    processed_text = torch.tensor(texto_pipeline(_text), dtype=torch.int64)
    text_list.append(processed_text)
    offset.append(processed_text.size(0))

  label_list = torch.tensor(label_list, dtype=torch.int64)
  offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)
  text_list = torch.cat(text_list)

  return label_list.to(device), text_list.to(device), offsets.to(device)

In [ ]:
from torch.utils.data import DataLoader

dataloader = DataLoader(
    list(test_iter),
    batch_size=8,
    shuffle=False,
    collate_fn=collate_batch
)
len(dataloader)

8750